In [1]:
# run if necessary
#%pip install numpy scipy matplotlib
#%pip install torch torchvision

# this file is mostly junk/scratchwork ish

In [2]:
import numpy as np

In [3]:
rows,cols = 5,7
B = np.random.choice([0,1], size=(rows,cols), p=[0.6,0.4])

In [4]:
def normalized_biadjacency(B):
    B = B.astype(float)
    deg_row = B.sum(axis=1)
    deg_col = B.sum(axis=0)

    deg_row[deg_row == 0] = 1
    deg_col[deg_col == 0] = 1

    row_scaling = 1 / np.sqrt(deg_row).reshape(-1, 1)  # shape: (rows, 1)
    col_scaling = 1 / np.sqrt(deg_col).reshape(1, -1)  # shape: (1, cols)

    B_tilde = B * row_scaling * col_scaling
    
    return(B_tilde)

def build_block_matrix(B):
    m,n = B.shape
    A = np.block([[np.zeros((m,m)),B],
                   [B.T, np.zeros((n,n))]])
    return A

def filter(x): #replace this one with anything
    return np.exp(-x)
def h(x): #even part
    return 0.5*(filter(x) + filter(-x))
def g(x): #odd part
    return 0.5*(filter(x) - filter(-x))


In [5]:
B_tilde = normalized_biadjacency(B)

In [6]:
B_tilde

array([[0.        , 0.40824829, 0.57735027, 0.        , 0.33333333,
        0.        , 0.        ],
       [0.5       , 0.        , 0.        , 0.70710678, 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.40824829, 0.        , 0.        , 0.33333333,
        0.57735027, 0.        ],
       [0.5       , 0.        , 0.        , 0.        , 0.40824829,
        0.        , 0.        ]])

In [7]:
A = build_block_matrix(B_tilde)
print(A)

[[0.         0.         0.         0.         0.         0.
  0.40824829 0.57735027 0.         0.33333333 0.         0.        ]
 [0.         0.         0.         0.         0.         0.5
  0.         0.         0.70710678 0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.40824829 0.         0.         0.33333333 0.57735027 0.        ]
 [0.         0.         0.         0.         0.         0.5
  0.         0.         0.         0.40824829 0.         0.        ]
 [0.         0.5        0.         0.         0.5        0.
  0.         0.         0.         0.         0.         0.        ]
 [0.40824829 0.         0.         0.40824829 0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.57735027 0.         0.         0.         0.         0.
  0.         0.         0.         

In [8]:
eigs, vects = np.linalg.eig(A)
v = np.diag(eigs)
u = vects
u_inv = np.linalg.inv(vects)

In [9]:
eigs = filter(eigs)
v = np.diag(eigs)
u @ v @ u_inv

array([[ 1.32568232e+00,  1.51720161e-03,  0.00000000e+00,
         1.54334279e-01,  7.57201495e-02, -1.22489672e-02,
        -4.71607900e-01, -6.38459209e-01, -2.10479662e-04,
        -3.94945957e-01, -2.84950792e-02,  0.00000000e+00],
       [ 1.51720161e-03,  1.40180596e+00,  0.00000000e+00,
         1.51720161e-03,  1.37558310e-01, -5.87236035e-01,
        -2.43040979e-04, -1.71855924e-04, -7.99254911e-01,
        -1.82246192e-02, -1.71855924e-04,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.54334279e-01,  1.51720161e-03,  0.00000000e+00,
         1.32568232e+00,  7.57201495e-02, -1.22489672e-02,
        -4.71607900e-01, -2.84950792e-02, -2.10479662e-04,
        -3.94945957e-01, -6.38459209e-01,  0.00000000e+00],
       [ 7.57201495e-02,  1.37558310e-01,  0.0000000

In [10]:
U,S,Vh = np.linalg.svd(B_tilde, full_matrices=True)
Sigma = np.zeros(B_tilde.shape)
print(S)
S = g(S)
np.fill_diagonal(Sigma,S)
U@Sigma@Vh


[1.00000000e+00 9.04315372e-01 5.77350269e-01 4.87615898e-01
 3.13764155e-17]


array([[-1.22489672e-02, -4.71607900e-01, -6.38459209e-01,
        -2.10479662e-04, -3.94945957e-01, -2.84950792e-02,
         0.00000000e+00],
       [-5.87236035e-01, -2.43040979e-04, -1.71855924e-04,
        -7.99254911e-01, -1.82246192e-02, -1.71855924e-04,
         0.00000000e+00],
       [-3.50073028e-17,  7.19628202e-17,  2.39500834e-17,
         1.07163355e-16, -3.16967384e-17,  7.78207129e-17,
         0.00000000e+00],
       [-1.22489672e-02, -4.71607900e-01, -2.84950792e-02,
        -2.10479662e-04, -3.94945957e-01, -6.38459209e-01,
         0.00000000e+00],
       [-5.57961439e-01, -1.97594387e-02, -1.39720331e-02,
        -3.12222545e-02, -4.53680944e-01, -1.39720331e-02,
         0.00000000e+00]])

In [11]:
filtered_A = np.block([[]])

ValueError: List at arrays[0] cannot be empty